# How to Build Image Generation Apps (Azure OpenAI)

LLM dem no only dey do text generation. You fit also generate image from text wey you describe. Images as modality useful well well for MedTech, architecture, tourism, game development, marketing, plus plenty other tins dem. For dis lesson, we go work with today **GPT Image** models wey dey for Microsoft Foundry.

## Wetin you go learn

By the time dis lesson finish, you go fit:

- Talk wetin image generation mean and where e fit work well.
- Understand di `gpt-image` model family and how e different from the old DALL·E models.
- Build app wey go generate image and edit image dem.

## Wetin be image generation?

Image generation models dey create images from text prompt. Modern models like `gpt-image` dey learn di connection between text and image while una dey train, then dem go convert random noise to image wey match di text you put.

Two image model families wey plenty people sabi na:

- **`gpt-image` (OpenAI)** — na di current generation wey we dey use for dis lesson. E fit do text-to-image generation and image editing (inpainting with mask).
- **Midjourney** — na popular model wey no be OpenAI, e get im own service plus im own Discord workflow.

> Di old OpenAI image models — **DALL·E 2** and **DALL·E 3** — na old models. DALL·E 3 no dey available again for new deployment dem, and only DALL·E 2 get features like `create_variation`. Make you use `gpt-image` models for new apps.

For Microsoft Foundry, **`gpt-image-2`** na di latest and best image model and na di one we recommend. `gpt-image-1.5` and `gpt-image-1-mini` still dey available for general use.

> **Important:** `gpt-image` models go return di image as **base64** (`b64_json`), no be URL. Your code go decode di base64 string to bytes then e go save am — no get image URL to download.


## Di way to build your first image generation application

So wetin e go take to build image generation application? You need dis kine libraries:

- **python-dotenv**, e good make you use dis library to keep your secret dem for *.env* file wey no dey inside the code.
- **openai**, dis library na wetin you go use take interact with OpenAI API.
- **pillow**, to work wit images for Python.

If you never do am before, follow the instruction for [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) page to create Microsoft Foundry resource and model. Choose **gpt-image-2** as the model (na di latest Azure OpenAI image model; DALL·E don old).

1. Create *.env* file wey get dis kind content:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    You fit find dis information for Microsoft Foundry portal for your resource inside the "Deployments" section.



1. Gather di libraries wey dey up for one file wey dem go call *requirements.txt* like dis:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Next, create one virtual environment and install di libraries:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, use di commands below to create and activate your virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add dis code for file wey dem call *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # configure Azure OpenAI (Microsoft Foundry) client.
    # Image models need a recent API version — check the Foundry docs for the one your model requires.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Put your prompt text here
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # e.g. "gpt-image-2"
        )
        # Set the folder wey go hold the image
        image_dir = os.path.join(os.curdir, 'images')

        # If the folder no dey, make am
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Start the image path (make sure say the file type na png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models go return the image as base64 (b64_json), no be URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Show the image for the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except BadRequestError as err:
        print(err)

    ```

Make we explain dis code:

- First, we import di libraries wey we need, including di OpenAI library, di dotenv library, di base64 module, and di Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Next, we load di environment variables from di *.env* file.

    ```python
    # import dotenv
    dotenv.load_dotenv()
    ```

- After dat, we configure di Azure OpenAI (Microsoft Foundry) client.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Next, we go generate di image:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Put your prompt text for here
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` models go return di image as **base64** string for `data[0].b64_json`. We decode am to bytes and write am inside file — no URL dey to download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Last last, we open di image and use di regular image viewer to show am:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mo' details on how to generate di image

Make we look di parameters for `images.generate`:

- **prompt**, na di text prompt wey dem use to generate di image. Here e be "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, na di size wey dem generate di image (for example `1024x1024`, `1536x1024`, `1024x1536`, or `"auto"`).
- **n**, na number of images wey dem generate. Here we generate one.
- **model**, na your image model deployment name (for example `gpt-image-2`).

> Image models no dey take `temperature` parameter — na text-generation control be dat. To get different kain things, call di API again; to reduce diversity, make your prompt more specific.

## Additional abilities for image generation

You don see how to generate image with small Python lines. `gpt-image` models fit also **edit** image wey dey already. If you give am image, optional **mask** (wey dey mark di place wey you wan change), and prompt, you fit change part of image — for example, put hat for our bunny.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits dey return as base64 too
edited_image = base64.b64decode(response.data[0].b64_json)
```

Di base image get only di rabbit; di final image get di hat for top di rabbit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
